# Colab: local LLM -> JSON -> validator -> KOMPAS macro

Этот ноутбук рассчитан на Google Colab с GPU T4.

Что делает:

1. Клонирует проект с GitHub.
2. Ставит `llama-cpp-python` и `huggingface_hub`.
3. Скачивает GGUF-модель Qwen Coder с Hugging Face.
4. Генерирует JSON по тексту пользователя.
5. Проверяет JSON через `resolve_placements()` и `validate_plan()`.
6. Если есть ошибка, возвращает ошибку модели и просит исправить JSON.
7. Сохраняет JSON, `.m3m` и журнал попыток в Google Drive.

## 1. Проверка GPU

В Colab включи: `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
!nvidia-smi

## 2. Google Drive для логов и результатов

In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/json-cad-pipeline')
DRIVE_LOG_DIR = DRIVE_ROOT / 'logs'
DRIVE_JSON_DIR = DRIVE_ROOT / 'generated_json'
DRIVE_MACRO_DIR = DRIVE_ROOT / 'generated_macros'
DRIVE_MODEL_DIR = DRIVE_ROOT / 'models'

for path in [DRIVE_LOG_DIR, DRIVE_JSON_DIR, DRIVE_MACRO_DIR, DRIVE_MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

## 3. Клонирование проекта с GitHub

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/12Jun34/json-cad-pipeline.git'
PROJECT_ROOT = Path('/content/json-cad-pipeline')

if PROJECT_ROOT.exists():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)

print('Project root:', PROJECT_ROOT)

## 4. Установка зависимостей

Эта ячейка может выполняться долго, потому что `llama-cpp-python` иногда собирается под CUDA.

In [ ]:
%pip install -q --upgrade huggingface_hub
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q --upgrade --force-reinstall --no-cache-dir llama-cpp-python

## 5. Скачивание модели

По умолчанию берется `Qwen/Qwen2.5-Coder-14B-Instruct-GGUF` и квант `Q4_K_M`. Для T4 это хороший стартовый вариант.

Если будет нехватка памяти или слишком медленно, поменяй `MODEL_REPO` на `Qwen/Qwen2.5-Coder-7B-Instruct-GGUF`.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

MODEL_REPO = 'Qwen/Qwen2.5-Coder-14B-Instruct-GGUF'
PREFERRED_QUANT = 'q4_k_m'

repo_files = list_repo_files(MODEL_REPO)
gguf_files = [name for name in repo_files if name.lower().endswith('.gguf')]
candidates = [name for name in gguf_files if PREFERRED_QUANT in name.lower()]

if not candidates:
    raise RuntimeError(f'No {PREFERRED_QUANT} GGUF file found in {MODEL_REPO}. Available: {gguf_files[:10]}')

MODEL_FILE = candidates[0]
MODEL_PATH = Path(hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=str(DRIVE_MODEL_DIR),
    local_dir_use_symlinks=False,
))

print('Model repo:', MODEL_REPO)
print('Model file:', MODEL_FILE)
print('Model path:', MODEL_PATH)

## 6. Импорт CAD-пайплайна

In [ ]:
import json
import re
import sys
from datetime import datetime

SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from macro_generator import write_macro
from placement_resolver import resolve_placements
from validator import validate_plan

LOG_PATH = DRIVE_LOG_DIR / 'json_repair_log.jsonl'
SFT_PATH = DRIVE_LOG_DIR / 'sft_dataset.jsonl'

print('LOG_PATH:', LOG_PATH)

## 7. Загрузка локальной модели

In [ ]:
from llama_cpp import Llama

N_CTX = 8192
N_GPU_LAYERS = -1
TEMPERATURE = 0.15
MAX_TOKENS = 2500
MAX_REPAIR_ATTEMPTS = 3

llm = Llama(
    model_path=str(MODEL_PATH),
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    verbose=False,
)

def chat(messages, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    result = llm.create_chat_completion(
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return result['choices'][0]['message']['content']

## 8. Спецификация JSON для модели

In [ ]:
CAD_JSON_SPEC = '''
You generate JSON plans for a KOMPAS macro generator.
Return only valid JSON. Do not use Markdown. Do not explain anything.

Top-level object:
{
  "version": "0.1",
  "units": "mm",
  "part_name": "ascii_snake_case_name",
  "commands": [ ... ]
}

Allowed command types only:
- create_box
- cut_box
- create_prism
- cut_prism
- create_triangle
- cut_triangle
- create_cylinder
- cut_cylinder

Never create object-specific command types. Forbidden examples:
- create_birdhouse
- create_plate_with_holes
- create_screwdriver
- create_table

Semantic placed box:
{
  "id": "body",
  "type": "create_box",
  "size": [width_y, height_z, depth_x],
  "placement": {"origin": [x, y, z]}
}

Triangle prism attached on top of a box:
{
  "id": "roof",
  "type": "create_triangle",
  "size": [width_y, height_z, depth_x],
  "attach": {"target": "body", "face": "top", "position": "center"}
}

Cylinder or cylindrical cut attached to front face:
{
  "id": "hole",
  "type": "cut_cylinder",
  "radius": 10,
  "depth": "half",
  "attach": {
    "target": "body",
    "face": "front",
    "position": "center",
    "offset": [dy, dz]
  }
}

Extrude modes:
- normal
- reverse
- middle

Rules:
- Use millimeters.
- Use stable ASCII ids.
- Every command id must be unique.
- A command can attach only to an object created earlier.
- Prefer placement/attach when the user describes relative positioning.
- Use low-level coordinates when the user gives exact coordinates.
- For a protruding peg/perch on the front face, use create_cylinder with extrude="reverse".
- For an inward hole on the front face, use cut_cylinder with extrude="normal".
'''.strip()

## 9. Примеры и память удачных генераций

In [ ]:
SEED_EXAMPLE_PATHS = [
    PROJECT_ROOT / 'examples' / 'birdhouse_attach.json',
    PROJECT_ROOT / 'examples' / 'plate_four_holes.json',
    PROJECT_ROOT / 'examples' / 'primitives_demo.json',
]

def load_json_file(path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def load_seed_examples():
    examples = []
    for path in SEED_EXAMPLE_PATHS:
        data = load_json_file(path)
        if data is not None:
            examples.append(data)
    return examples

def load_memory_examples(limit=3):
    if not LOG_PATH.exists():
        return []
    records = []
    for line in LOG_PATH.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        if record.get('success') and record.get('final_plan'):
            records.append(record['final_plan'])
    return records[-limit:]

def format_examples(examples, limit=4):
    if not examples:
        return 'No examples yet.'
    chunks = []
    for index, example in enumerate(examples[-limit:], start=1):
        chunks.append(f'Example {index}:\n' + json.dumps(example, ensure_ascii=False, indent=2))
    return '\n\n'.join(chunks)

## 10. Проверка JSON и сохранение результата

In [ ]:
def extract_json_object(text):
    cleaned = text.strip()
    fence_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', cleaned, flags=re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1)
    first = cleaned.find('{')
    last = cleaned.rfind('}')
    if first == -1 or last == -1 or last <= first:
        raise ValueError('Model output does not contain a JSON object')
    return json.loads(cleaned[first:last + 1])

def validate_and_resolve(plan):
    resolved_plan, was_resolved = resolve_placements(plan)
    validate_plan(resolved_plan)
    return resolved_plan, was_resolved

def save_generated_outputs(raw_plan, resolved_plan):
    part_name = resolved_plan['part_name']
    raw_json_path = DRIVE_JSON_DIR / f'{part_name}.json'
    resolved_json_path = DRIVE_JSON_DIR / f'{part_name}.resolved.json'
    raw_json_path.write_text(json.dumps(raw_plan, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    resolved_json_path.write_text(json.dumps(resolved_plan, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    macro_path = write_macro(resolved_plan, DRIVE_MACRO_DIR)
    return {
        'raw_json_path': str(raw_json_path),
        'resolved_json_path': str(resolved_json_path),
        'macro_path': str(macro_path),
    }

## 11. Промпты генерации и ремонта

In [ ]:
def build_system_message():
    return (
        'You are a CAD JSON planner. '
        'You output only valid JSON for the provided schema. '
        'You never invent object-specific commands.'
    )

def build_generation_messages(user_request, part_name):
    examples = load_seed_examples() + load_memory_examples(limit=3)
    user_message = f'''
{CAD_JSON_SPEC}

Known good examples:
{format_examples(examples, limit=4)}

User request in Russian:
{user_request}

Use this part_name: {part_name}

Return only the JSON plan.
'''.strip()
    return [
        {'role': 'system', 'content': build_system_message()},
        {'role': 'user', 'content': user_message},
    ]

def build_repair_messages(user_request, part_name, bad_output, error):
    user_message = f'''
{CAD_JSON_SPEC}

Original user request:
{user_request}

Required part_name: {part_name}

Your previous output was invalid:
{bad_output}

Validator error:
{error}

Fix the JSON. Return only one complete valid JSON object.
'''.strip()
    return [
        {'role': 'system', 'content': build_system_message()},
        {'role': 'user', 'content': user_message},
    ]

## 12. Цикл: генерация -> ошибка -> исправление

In [ ]:
def append_log(record):
    with LOG_PATH.open('a', encoding='utf-8') as file:
        file.write(json.dumps(record, ensure_ascii=False) + '\n')

def generate_plan_with_repair(user_request, part_name, max_attempts=MAX_REPAIR_ATTEMPTS):
    attempts = []
    messages = build_generation_messages(user_request, part_name)

    for attempt_index in range(1, max_attempts + 1):
        output = chat(messages)
        attempt = {'attempt': attempt_index, 'raw_output': output}

        try:
            raw_plan = extract_json_object(output)
            resolved_plan, was_resolved = validate_and_resolve(raw_plan)
            paths = save_generated_outputs(raw_plan, resolved_plan)

            attempt.update({'ok': True, 'was_resolved': was_resolved, 'paths': paths})
            attempts.append(attempt)

            record = {
                'created_at': datetime.now().isoformat(timespec='seconds'),
                'success': True,
                'user_request': user_request,
                'part_name': part_name,
                'attempts': attempts,
                'final_plan': raw_plan,
                'resolved_plan': resolved_plan,
                'paths': paths,
            }
            append_log(record)
            return record

        except Exception as error:
            attempt.update({'ok': False, 'error': str(error)})
            attempts.append(attempt)
            messages = build_repair_messages(user_request, part_name, output, str(error))

    record = {
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'success': False,
        'user_request': user_request,
        'part_name': part_name,
        'attempts': attempts,
        'final_plan': None,
    }
    append_log(record)
    return record

## 13. Тестовый запуск

In [ ]:
USER_REQUEST = '''
Сделай скворечник: прямоугольный корпус, сверху треугольная крыша, в центре передней стороны круглое отверстие до середины корпуса, под отверстием жердочка цилиндром наружу.
'''.strip()

PART_NAME = 'colab_birdhouse_test'

record = generate_plan_with_repair(USER_REQUEST, PART_NAME)
print(json.dumps(record, ensure_ascii=False, indent=2))

## 14. SFT-датасет для будущего дообучения

Эта ячейка превращает успешные записи из лога в `jsonl`-датасет формата chat messages.

In [ ]:
def build_sft_dataset():
    if not LOG_PATH.exists():
        return 0

    count = 0
    with LOG_PATH.open('r', encoding='utf-8') as source, SFT_PATH.open('w', encoding='utf-8') as target:
        for line in source:
            if not line.strip():
                continue
            record = json.loads(line)
            if not record.get('success') or not record.get('final_plan'):
                continue
            sample = {
                'messages': [
                    {'role': 'system', 'content': build_system_message()},
                    {'role': 'user', 'content': CAD_JSON_SPEC + '\n\nUser request:\n' + record['user_request']},
                    {'role': 'assistant', 'content': json.dumps(record['final_plan'], ensure_ascii=False)},
                ]
            }
            target.write(json.dumps(sample, ensure_ascii=False) + '\n')
            count += 1
    return count

count = build_sft_dataset()
print(f'SFT samples written: {count} -> {SFT_PATH}')